In [11]:
import json
import re
from pathlib import Path
from typing import Dict, List, Any
from collections import Counter


***SCENE JSON BUILDER***

In [12]:
def parse_objects(path_objects: Path, fps: float = 25.0) -> Dict[int, Dict[str, Any]]:
    type_labels: Dict[int, str] = {1: "person", 2: "car", 3: "vehicles", 4: "object", 5: "bike"}
    objects: Dict[int, Dict[str, Any]] = {}

    min_frames: Dict[int, int] = {}
    max_frames: Dict[int, int] = {}
    type_ids: Dict[int, int] = {}

    for line in path_objects.open():
        cols = line.split()
        oid = int(cols[0])
        frame = int(cols[2])
        type_id = int(cols[7])
        type_ids[oid] = type_id

        if oid not in min_frames or frame < min_frames[oid]:
            min_frames[oid] = frame
        if oid not in max_frames or frame > max_frames[oid]:
            max_frames[oid] = frame

    for oid in min_frames:
        type_id = type_ids[oid]
        objects[oid] = {
            "object_id": oid,
            "type_id": type_id,
            "type_label": type_labels.get(type_id, "unknown"),
            "start_offset_sec": min_frames[oid] / fps,
            "end_offset_sec": max_frames[oid] / fps
        }

    return objects


In [13]:
def parse_events(path_events: Path, fps: float = 25.0) -> Dict[int, Dict[str, Any]]:
    event_labels: Dict[int, str] = {
        1: "Person loading an Object to a Vehicle",
        2: "Person Unloading an Object from a Vehicle",
        3: "Person Opening a Vehicle Trunk",
        4: "Person Closing a Vehicle Trunk",
        5: "Person getting into a Vehicle",
        6: "Person getting out of a Vehicle",
        7: "Person gesturing",
        8: "Person digging",
        9: "Person carrying an object",
        10: "Person running",
        11: "Person entering a facility",
        12: "Person exiting a facility"
    }
    events: Dict[int, Dict[str, Any]] = {}
    for line in path_events.open():
        cols = line.split()
        eid = int(cols[0])
        type_id = int(cols[1])
        start_frame = int(cols[3])
        end_frame = int(cols[4])
        ev = events.setdefault(eid, {
            "event_id": eid,
            "type_id": type_id,
            "type_label": event_labels.get(type_id, "unknown"),
            "start_frame": start_frame,
            "end_frame": end_frame,
            "start_offset_sec": start_frame / fps,
            "end_offset_sec": end_frame / fps
        })
    return events

In [14]:

def parse_mapping(path_map: Path) -> Dict[int, List[int]]:
    assoc: Dict[int, List[int]] = {}
    for line in path_map.open():
        cols = line.split()
        eid = int(cols[0])
        num_obj = int(cols[5])
        flags = cols[6:6+num_obj]
        obj_ids = [i+1 for i, flag in enumerate(flags) if flag == '1']
        assoc[eid] = obj_ids
    return assoc

***DETECTION COUNTS AND PRIORITY ASSIGNMENT***

In [15]:
def count_detections_per_scene(base_dir: Path) -> Dict[str, int]:
    counts: Dict[str, int] = {}
    for obj_file in base_dir.glob("VIRAT_S_*_*_*.viratdata.objects.txt"):
        parts = obj_file.name.split("_")
        scene_id = "_".join(parts[:2]) + "_" + parts[2][:4]
        counts.setdefault(scene_id, 0)
        with obj_file.open() as f:
            counts[scene_id] += sum(1 for _ in f)
    return counts


***SCENE JSON BUILDER***

In [16]:
def build_scene_json(scene_id: str, base_dir: Path, output_dir: Path, fps: float = 25.0) -> None:
    scene: Dict[str, Any] = {
        "scene_id": scene_id,
        "subclips": []
    }

    for obj_file in base_dir.glob("VIRAT_S_*_*_*.viratdata.objects.txt"):
        parts = obj_file.name.split("_")
        scene_prefix = "_".join(parts[:2]) + "_" + parts[2][:4]
        if scene_prefix != scene_id:
            continue

        clip_id = obj_file.name.replace(".viratdata.objects.txt", "")
        match = re.search(r"_(\d{6})_(\d{6})$", clip_id)
        if not match:
            continue
        start_sec, end_sec = int(match.group(1)), int(match.group(2))

        try:
            objs = parse_objects(base_dir / f"{clip_id}.viratdata.objects.txt", fps)
            evs = parse_events(base_dir / f"{clip_id}.viratdata.events.txt", fps)
            mapping = parse_mapping(base_dir / f"{clip_id}.viratdata.mapping.txt")
        except FileNotFoundError:
            continue

        for eid, obj_ids in mapping.items():
            if eid in evs:
                evs[eid]["object_ids"] = obj_ids

        token_counter: Counter = Counter()

        for o in objs.values():
            for word in o["type_label"].lower().split():
                token_counter[word] += 1

        for e in evs.values():
            for word in e["type_label"].lower().split():
                token_counter[word] += 1

        subclip: Dict[str, Any] = {
            "clip_id": clip_id,
            "video_filename": f"{clip_id}.mp4",
            "start_sec": start_sec,
            "end_sec": end_sec,
            "textual_tokens": dict(token_counter),
            "events": list(evs.values()),
            "objects": list(objs.values())
        }
        scene["subclips"].append(subclip)

    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / f"{scene_id}.json"
    with open(output_file, "w") as fw:
        json.dump(scene, fw, indent=2)
    print(f"✅ JSON generado sin prioridad: {output_file}")

***EJECUCIÓN GENERAL***

In [17]:
def run_pipeline_with_output(base_dir: Path, output_dir: Path) -> None:
    counts = count_detections_per_scene(base_dir)
    scene_ids = list(counts.keys())
    for scene_id in scene_ids:
        build_scene_json(scene_id, base_dir, output_dir)

In [18]:
from pathlib import Path
# Carpeta que contiene todos los .viratdata.*.txt
input_dir = Path("annotations/")

# Carpeta donde quieres guardar los .json
output_dir = Path("json/")
run_pipeline_with_output(input_dir, output_dir)



✅ JSON generado sin prioridad: json/VIRAT_S_0100.json
✅ JSON generado sin prioridad: json/VIRAT_S_0101.json
✅ JSON generado sin prioridad: json/VIRAT_S_0502.json
✅ JSON generado sin prioridad: json/VIRAT_S_0102.json
✅ JSON generado sin prioridad: json/VIRAT_S_0002.json
✅ JSON generado sin prioridad: json/VIRAT_S_0400.json
✅ JSON generado sin prioridad: json/VIRAT_S_0503.json
✅ JSON generado sin prioridad: json/VIRAT_S_0401.json
✅ JSON generado sin prioridad: json/VIRAT_S_0500.json


In [19]:
# hacer una funcion que liste todos los archivos .json de la carpeta llamada json
# def printFolder(pathFolder: Path) -> None:
#     if not pathFolder.exists():
#         print(f"El directorio {pathFolder} no existe.")
#         return
#     json_files = list(pathFolder.glob("*.json"))
#     if not json_files:
#         print(f"No se encontraron archivos JSON en {pathFolder}.")
#     else:
#         print(f"Archivos JSON encontrados en {pathFolder}:")
#         for file in json_files:
#             print(file.name)


# printFolder(output_dir)